
##### The 5 Rules of Tool Descriptions
1. What the Tool Does: Start with a clear, one-sentence statement of the tool's purpose. Use active voice. Be specific about the domain it operates in
2. When to Use It: Explicitly state the conditions under which this tool should be selected.
3. Input Format & Constraints: Document each parameter's type, format, valid range, and any special encoding requirements. JSON Schema handles type validation, but Claude needs human-readable context for semantic correctness.
4. Output Shape: Describe what the tool returns, including the structure, key fields, and any pagination behavior. Claude needs this to plan its next steps before the tool even runs.
5. Edge Cases & Gotchas. Document known failure modes, rate limits, and surprising behaviors. This is what separates a "good enough" description from an excellent one.

// Good: specific domain + action
"Retrieves a customer's billing history from Stripe, returning a list of invoices with amounts dates, and payment status."

"Use this tool when the user asks about past payments, invoice history, or billing charges. Do NOT use for updating payment methods -- use update_payment_method instead."

"customer_id: The Stripe customer ID (starts with'cus_'). Required. If the user provides an email instead, use lookup_customer first."
"date_range: ISO 8601 date range. Max span is 12 months. If not provided, defaults to last 30 days."

"Returns { invoices: Invoice[], hasMore: boolean, cursor: string }. 
Each Invoice includes id, amount (in cents), currency, status ('paid'|'open'|'void'), and created_at. Results are paginated; pass cursor to get the next page."

"Returns an empty array (not an error) if the customer has no billing history. Rate-limited to 100 requests per minute. If the customer was deleted, returns { error: 'customer_archived' } -- this is NOT a permission error; suggest the user contact support."

##### Error Handling Mastery
###### Error Taxonomy. Categories
1. Transient Errors: Temporary failures caused by network issues, rate limits, or service unavailability. Claude should retry after a delay, ideally with exponential backoff.
2. Validation Errors: The tool was called with invalid parameters. Claude should fix the input and retry -- never retry with the same parameters.
3. Business Logic Errors: The request is technically valid but violates a business rule. Claude should explain the constraint to the user and suggest alternatives.
4. Permission Errors: The caller lacks authorization. Claude should not retry and should inform the user about the required permissions. 

###### Structured Error Metadata. 5 Key metadata fields
{
  "isError": true,
  "errorCategory": "validation",
  "isRetryable": false,
  "description": "The customer_id 'usr_123' is not a valid Stripe customer ID. Stripe customer IDs start with 'cus_'. Did you mean to use lookup_customer with the user ID first?",
  "recoveryGuidance": "Call lookup_customer with the user ID to get the Stripe customer ID, then  retry this call."
}



##### MCP Server Patterns
###### Configuration Structure
{
  "mcpServers": {
    "database": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-postgres"],
      "env": {
        "DATABASE_URL": "${DB_CONNECTION_STRING}"
      }
    },
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": {
        "GITHUB_TOKEN": "${GITHUB_TOKEN}"
      }
    }
  }
}

// Instead of Claude calling get_table_schema("users")
// every time, expose it as a resource:
{
  "resources": [
    {
      "uri": "schema://tables/users",
      "name": "Users Table Schema",
      "description": "Column definitions for the users table",
      "mimeType": "application/json"
    }
  ]
}

